In [ ]:
!pip install arxiv sentence-transformers -q

In [ ]:
import pandas as pd
import arxiv

papers = []

client = arxiv.Client()

search = arxiv.Search(
    query="artificial intelligence",
    max_results=500
)

for result in client.results(search):
    papers.append({
        "title": result.title,
        "summary": result.summary,
        "authors": ", ".join(
            [author.name for author in result.authors]
        )
    })

df = pd.DataFrame(papers)

df.to_csv("cleaned_arxiv_papers.csv", index=False)

print("Dataset created successfully")
print(df.head())

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = model.encode(
    df["summary"].tolist(),
    show_progress_bar=True
)

np.save("arxiv_embeddings.npy", embeddings)

print("Embeddings saved:", embeddings.shape)

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned_arxiv_papers.csv")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(
    df["summary"].tolist(),
    show_progress_bar=True
)

np.save("arxiv_embeddings.npy", embeddings)

print("Embeddings saved!")

In [ ]:
print(embeddings.shape)
print(embeddings.dtype)
embeddings

In [ ]:
!pip install faiss-cpu -q

In [ ]:
import faiss

# 1. We must make a copy of the embeddings first because FAISS normalizes "in-place"
# (Meaning it permanently alters the original variable)
faiss_embeddings = embeddings.copy()

# 2. Apply L2 Normalization
faiss.normalize_L2(faiss_embeddings)

print("Embeddings normalized! Ready for the FAISS Index.")

In [ ]:
# index = faiss.IndexFlatIP(384)
# index.add(faiss_embeddings)
# index

In [ ]:
import os
import faiss

index_path = "paper_faiss.index"

if os.path.exists(index_path):
    print("Loading existing FAISS index")
    index = faiss.read_index(index_path)

else:
    print("Creating new FAISS index")

    faiss_embeddings = embeddings.copy().astype('float32')

    faiss.normalize_L2(faiss_embeddings)

    dimension = faiss_embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)

    index.add(faiss_embeddings)

    faiss.write_index(index,index_path)

    print("FAISS index saved")

In [ ]:
query = "deep learning for medical image analysis"
query_embedding = model.encode(query)
print(query_embedding.shape)

In [ ]:
query_embedding = model.encode([query])
# query_embedding = model.encode(query).reshape(1,-1)
print(query_embedding.shape)
print(query_embedding)

In [ ]:
faiss.normalize_L2(query_embedding)
query_embedding

In [ ]:
D, I = index.search(query_embedding, 5)

print("Scores (D):", D)
print("Indices (I):", I)

In [ ]:
print(df.shape)

In [ ]:
print(df.head())
print(df.tail())

In [ ]:
random_idx = np.random.randint(0, len(df))

print("Index:", random_idx)
print("Title:", df.iloc[random_idx]["title"])

In [ ]:
!pip install keybert -q

In [ ]:
!pip install faiss-cpu keybert transformers sentence-transformers scikit-learn arxiv -q

In [ ]:
import faiss
from transformers import pipeline
from keybert import KeyBERT
import numpy as np

# Build/load FAISS index
faiss_embeddings = embeddings.copy().astype("float32")
faiss.normalize_L2(faiss_embeddings)

dimension = faiss_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(faiss_embeddings)

# Safe row display (instead of 26063)
print(df.iloc[10]["title"])


# ---------------- SEARCH FUNCTION ----------------

def search_paper(query, k=5):

    query_embedding = model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    D,I=index.search(query_embedding,k)

    for score,idx in zip(D[0],I[0]):

        print("Similarity Score:",score)

        print(
            "Title:",
            df.iloc[idx]["title"]
        )

        print(
            "Summary:",
            df.iloc[idx]["summary"][:500]
        )

        print()

    return D,I


query="deep learning for medical image analysis"

D,I=search_paper(query)


# ---------------- SUMMARIZER ----------------

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load summarization model
model_name = "sshleifer/distilbart-cnn-12-6"

tokenizer = AutoTokenizer.from_pretrained(model_name)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Test summary on one paper
text = df.iloc[10]["summary"]

inputs = tokenizer(
    text,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

summary_ids = summary_model.generate(
    inputs["input_ids"],
    max_length=120,
    min_length=40
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print(summary)


# SEARCH + SUMMARY
def search_and_summarize(query, k=5):

    query_embedding = model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    D,I=index.search(query_embedding,k)

    for score,idx in zip(D[0],I[0]):

        print("Similarity:",score)

        print("Title:",df.iloc[idx]["title"])

        print("Summary Preview:")
        print(df.iloc[idx]["summary"][:500])

        text=df.iloc[idx]["summary"]

        inputs=tokenizer(
            text,
            return_tensors="pt",
            max_length=1024,
            truncation=True
        )

        summary_ids=summary_model.generate(
            inputs["input_ids"],
            max_length=120,
            min_length=40
        )

        summary=tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        print("\nAI Summary:")
        print(summary)

        print("-"*80)


search_and_summarize(
    "Deep learning in medical science"
)

In [ ]:
def search_paper(query,k=5):
    query_embedding=model.encode([query])
    faiss.normalize_L2(query_embedding)
    D,I=index.search(query_embedding,k)
    for score,idx in zip(D[0],I[0]):
        print("Similarity Score: ",score)
        print("Title: ",df.iloc[idx]['title'])
        print("Abstract:" ,df.iloc[idx]['abstract'][:500])
        print() # for space between two papers
    return D,I

In [ ]:
print(query)

In [ ]:
print(search_paper)

In [ ]:
df["abstract"] = df["summary"]

print(df.columns)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "sshleifer/distilbart-cnn-12-6"

tokenizer = AutoTokenizer.from_pretrained(model_name)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def summarizer(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    summary_ids = summary_model.generate(
        inputs["input_ids"],
        max_length=120,
        min_length=40
    )

    return [{
        "summary_text":
        tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )
    }]

In [ ]:
for score, idx in zip(D[0], I[0]):

    print("Title:", df.iloc[idx]["title"])

    summary = summarizer(
        df.iloc[idx]["abstract"]
    )

    print("AI SUMMARY:")
    print(summary[0]["summary_text"])

    print("-"*80)

In [ ]:
print(type(index))
print(index.ntotal)
D, I = search_paper(query)

In [ ]:
from keybert import KeyBERT

# ---------------- SUMMARIZER ----------------

from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

model_name="sshleifer/distilbart-cnn-12-6"

tokenizer=AutoTokenizer.from_pretrained(
    model_name
)

summary_model=AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)


def summarizer(text):

    inputs=tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    summary_ids=summary_model.generate(
        inputs["input_ids"],
        max_length=120,
        min_length=40
    )

    return [{
        "summary_text":
        tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )
    }]


# Test one paper

summary=summarizer(
    df.iloc[10]["abstract"]
)

print(summary[0]["summary_text"])


# ---------------- SEARCH + SUMMARY ----------------

def search_and_summarize(query,k=5):

    query_embedding=model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    D,I=index.search(query_embedding,k)

    for score,idx in zip(D[0],I[0]):

        print(
            "Similarity Score:",
            score
        )

        print(
            "Title:",
            df.iloc[idx]["title"]
        )

        print(
            "Abstract Preview:"
        )

        print(
            df.iloc[idx]["abstract"][:500]
        )

        print()

        summary=summarizer(
            df.iloc[idx]["abstract"]
        )

        print(
            "AI SUMMARY:"
        )

        print(
            summary[0]["summary_text"]
        )

        print("-"*80)


search_and_summarize(
    "Deep learning in medical science"
)



# ---------------- KEYWORDS ----------------

kw_model=KeyBERT(model=model)

text=df.iloc[10]["abstract"]

keywords=kw_model.extract_keywords(
    text
)

print(keywords)

finalkeyword=kw_model.extract_keywords(
    text,
    keyphrase_ngram_range=(1,3),
    stop_words=None
)

print(finalkeyword)

In [ ]:
!pip install spacy -q
!python -m spacy download en_core_web_sm

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_entities(text):

    doc=nlp(text)

    entities=[]

    for ent in doc.ents:
        entities.append(
            (ent.text,ent.label_)
        )

    return entities


text=df.iloc[10]["abstract"]

entities=extract_entities(text)

print(entities)

In [ ]:
!pip install rank-bm25 -q

In [ ]:
from rank_bm25 import BM25Okapi

corpus=df["abstract"].tolist()

tokenized_corpus=[
    doc.split()
    for doc in corpus
]

bm25=BM25Okapi(
    tokenized_corpus
)


def hybrid_search(query,k=5):

    query_tokens=query.split()

    bm25_scores=bm25.get_scores(
        query_tokens
    )

    query_embedding=model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(
        query_embedding
    )

    D,I=index.search(
        query_embedding,
        len(df)
    )

    semantic_scores=np.zeros(
        len(df)
    )

    semantic_scores[I[0]]=D[0]

    final_scores=(
        0.5*bm25_scores+
        0.5*semantic_scores
    )

    top=np.argsort(
        final_scores
    )[-k:][::-1]

    for idx in top:

        print(
            df.iloc[idx]["title"]
        )

        print(
            final_scores[idx]
        )
        print()


hybrid_search(
    "medical image segmentation"
)

In [ ]:
from transformers import pipeline

generator=pipeline(
    "text-generation",
    model="gpt2"
)

def rag(query):

    D,I=search_paper(
        query,
        k=3
    )

    context=" ".join(
        [
        df.iloc[idx]["abstract"]
        for idx in I[0]
        ]
    )

    prompt=f"""
    Context:{context}

    Question:{query}

    Answer:
    """

    answer=generator(
        prompt,
        max_new_tokens=100
    )

    print(
        answer[0]["generated_text"]
    )

rag(
    "Recent methods for medical image segmentation"
)

In [ ]:
!pip install networkx matplotlib -q

In [ ]:
import spacy
import networkx as nx
import matplotlib.pyplot as plt

# Load NLP model
nlp = spacy.load("en_core_web_sm")

# Select paper text
text = df.iloc[10]["abstract"]

# Extract entities
doc = nlp(text)

entities=[]

for ent in doc.ents:
    entities.append(
        (ent.text,ent.label_)
    )

print("Extracted Entities:")
print(entities)

# Build graph
G=nx.Graph()

for ent,label in entities:
    G.add_node(
        ent,
        entity_type=label
    )

for i in range(len(entities)-1):

    G.add_edge(
        entities[i][0],
        entities[i+1][0]
    )

# Draw graph
plt.figure(figsize=(12,8))

nx.draw(
    G,
    with_labels=True
)

plt.show()

In [ ]:
def recommend_papers(
    paper_index,
    k=5
):

    paper_embedding=embeddings[
        paper_index
    ].reshape(
        1,-1
    ).astype(
        "float32"
    )

    D,I=index.search(
        paper_embedding,
        k+1
    )

    for idx in I[0][1:]:

        print(
            df.iloc[idx]["title"]
        )
        print()


recommend_papers(10)